In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd

/home/kxelina/RAG_project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialize embedding model and ChromaDB
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding model loaded: {embedding_model.get_sentence_embedding_dimension()} dimensions")

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(
    name="sales_data",
    metadata={"hnsw:space": "cosine"}
)
print(f"ChromaDB: {collection.name}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7935.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: 384 dimensions
ChromaDB: sales_data


In [3]:
# Store chunks and embeddings in ChromaDB
chunking_results = pd.read_pickle('chunking_analysis.pkl')
chunked_documents = chunking_results[1000]['documents']

batch_size = 100
total_chunks = len(chunked_documents)

for i in range(0, total_chunks, batch_size):
    batch = chunked_documents[i:i + batch_size]
    
    ids = [str(doc.metadata['chunk_id']) for doc in batch]
    documents = [doc.page_content for doc in batch]
    metadatas = [doc.metadata for doc in batch]
    
    # Generate embeddings
    embeddings = embedding_model.encode(documents).tolist()
    
    # Store in ChromaDB
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )

print(f"\nTotal chunks in database: {collection.count()}")


Total chunks in database: 2603


In [4]:
# Retrieval functions
def similarity_search(query, num_results=5, filters=None):
    query_embedding = embedding_model.encode(query).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=num_results,
        where=filters
    )
    
    return results

def search_by_metadata(category=None, region=None, year=None, num_results=10):
    filters = {}
    
    if category:
        filters['category'] = category
    if region:
        filters['region'] = region
    if year:
        filters['year'] = str(year)
    
    results = collection.get(
        where=filters if filters else None,
        limit=num_results
    )
    
    return results

def combined_search(query, category=None, region=None, year=None, num_results=5):
    query_embedding = embedding_model.encode(query).tolist()
    
    filters = {}
    if category:
        filters['category'] = category
    if region:
        filters['region'] = region
    if year:
        filters['year'] = str(year)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=num_results,
        where=filters if filters else None
    )
    
    return results

In [5]:
# Test retrieval
query = "What were the highest sales?"
results = similarity_search(query, num_results=3)

print(f"Query: {query}")
print(f"Results found: {len(results['documents'][0])}")
print()

for i, doc in enumerate(results['documents'][0]):
    print(f"Result {i+1}:")
    print(f"  Content: {doc[:150]}...")
    print(f"  Metadata: {results['metadatas'][0][i]}")
    print()


Query: What were the highest sales?
Results found: 3

Result 1:
  Content: On December 25, 2017, 7 units of Maxell 4.7GB DVD-R (Technology - Accessories) were sold in Fairfield, Ohio (East) for 158.928 dollars with a discount...
  Metadata: {'year': '2017', 'month': 'December', 'category': 'Technology', 'region': 'East', 'chunk_size': 858, 'chunk_id': 628}

Result 2:
  Content: On October 31, 2014, 9 units of Maxell LTO Ultrium - 800 GB (Technology - Accessories) were sold in Los Angeles, California (West) for 251.91 dollars,...
  Metadata: {'category': 'Technology', 'region': 'West', 'year': '2014', 'chunk_id': 535, 'chunk_size': 999, 'month': 'October'}

Result 3:
  Content: On October 23, 2015, 9 units of Maxell DVD-RAM Discs (Technology - Accessories) were sold in San Diego, California (West) for 148.32 dollars, resultin...
  Metadata: {'chunk_size': 808, 'year': '2015', 'category': 'Technology', 'chunk_id': 1006, 'region': 'West', 'month': 'October'}

